# 05 · Hypothesis Testing

Digital Onboarding Optimization Case Study

> Fully synthetic dataset. No real company, customer, or proprietary information is included.

## 1. Objective

This notebook tests whether the difference in approval rate between `upload_now` and `upload_later` users is statistically meaningful.

We will use:

- Two-proportion z-test
- Confidence intervals
- Chi-square test
- Effect size
- Odds ratio

The objective is not only statistical significance, but business significance.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

users = pd.read_csv(DATA_DIR / "onboarding_users.csv", parse_dates=["signup_timestamp"])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_timestamp"])
support = pd.read_csv(DATA_DIR / "support_calls.csv", parse_dates=["call_timestamp"])

users.head()

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

## 2. Main Hypothesis

In [ ]:
summary = users.groupby("upload_choice").agg(
    users=("user_id", "count"),
    approvals=("approved", "sum"),
    approval_rate=("approved", "mean")
)
summary

In [ ]:
now = users[users["upload_choice"] == "upload_now"]
later = users[users["upload_choice"] == "upload_later"]

count = np.array([now["approved"].sum(), later["approved"].sum()])
nobs = np.array([len(now), len(later)])

z_stat, p_value = proportions_ztest(count, nobs)

effect = now["approved"].mean() - later["approved"].mean()
relative_lift = now["approved"].mean() / later["approved"].mean() - 1

pd.DataFrame({
    "metric": ["z_statistic", "p_value", "absolute_difference", "relative_lift"],
    "value": [z_stat, p_value, effect, relative_lift]
})

## 3. Confidence Interval for Difference in Proportions

In [ ]:
ci_low, ci_high = confint_proportions_2indep(
    count1=int(now["approved"].sum()),
    nobs1=len(now),
    count2=int(later["approved"].sum()),
    nobs2=len(later),
    method="wald"
)

pd.DataFrame({
    "estimate": ["approval_rate_difference"],
    "lower_95_ci": [ci_low],
    "upper_95_ci": [ci_high]
})

## 4. Chi-square Test

In [ ]:
contingency = pd.crosstab(users["upload_choice"], users["approved"])
chi2, p, dof, expected = stats.chi2_contingency(contingency)

pd.DataFrame({
    "metric": ["chi_square", "p_value", "degrees_of_freedom"],
    "value": [chi2, p, dof]
})

## 5. Odds Ratio

In [ ]:
# Contingency table:
# upload_now approved / not approved
# upload_later approved / not approved

a = int(now["approved"].sum())
b = int(len(now) - now["approved"].sum())
c = int(later["approved"].sum())
d = int(len(later) - later["approved"].sum())

odds_ratio = (a / b) / (c / d)

pd.DataFrame({
    "metric": ["odds_ratio_upload_now_vs_later"],
    "value": [odds_ratio]
})

## 6. Support Contact Hypothesis

In [ ]:
users_support = users.copy()
users_support["has_support_call"] = users_support["user_id"].isin(support["user_id"]).astype(int)

support_summary = users_support.groupby("upload_choice").agg(
    users=("user_id", "count"),
    support_calls=("has_support_call", "sum"),
    support_rate=("has_support_call", "mean")
)
support_summary

In [ ]:
now_s = users_support[users_support["upload_choice"] == "upload_now"]
later_s = users_support[users_support["upload_choice"] == "upload_later"]

count_s = np.array([now_s["has_support_call"].sum(), later_s["has_support_call"].sum()])
nobs_s = np.array([len(now_s), len(later_s)])

z_stat_s, p_value_s = proportions_ztest(count_s, nobs_s)

pd.DataFrame({
    "metric": ["support_z_statistic", "support_p_value", "support_rate_difference"],
    "value": [z_stat_s, p_value_s, now_s["has_support_call"].mean() - later_s["has_support_call"].mean()]
})

## 7. Executive Conclusion

The statistical evidence supports the business hypothesis:

> Immediate document submission is associated with substantially higher approval conversion and lower operational friction.

This does not automatically prove causality, because the dataset is observational. However, the magnitude of the difference is large enough to justify a controlled A/B test focused on the document-submission screen.